# Fine-tuning Qwen3.5-2B for ABSA

In this notebook we train a LoRA adapter on the SemEval restaurant training split, then evaluate it.

The training prompt is the **same** zero-shot prompt the main notebook uses, so the adapter learns to answer exactly the input it will be served at inference time.

## 1. Runtime check and Drive

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU"
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
PROJECT  = Path("/content/drive/MyDrive/absa")
DATA_DIR = PROJECT / "data"
OUT_DIR  = PROJECT / "finetune" / "qwen3_5_2b"
PRED_DIR = PROJECT / "predictions"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)
print("train.json present:", (DATA_DIR / "train.json").exists())
print("test.json  present:", (DATA_DIR / "test.json").exists())

GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train.json present: True
test.json  present: True


## 2. Install Unsloth (Qwen3.5-specific)

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0

!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
import unsloth, transformers, trl
print("unsloth", unsloth.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth 2026.6.1 | transformers 5.2.0 | trl 0.22.2


## 3. Build instruction data from the split

`SYSTEM_INSTRUCTION` and `build_prompt` are copied from the main notebook so the prompt the adapter learns on is exactly the prompt served at inference time. Each training example becomes a
two-turn chat: the user turn is `build_prompt(sentence)`, the assistant turn is the gold aspect list
as compact JSON. Sentences with no aspect map to `[]`, teaching the model to return empty array instead of
hallucinating aspects.

In [ ]:
import json
SENTIMENTS = ["positive", "negative", "neutral", "conflict"]

SYSTEM_INSTRUCTION = (
    "You are an aspect-based sentiment analysis system for customer reviews.\n"
    "Given one sentence, extract every aspect term that is explicitly mentioned and its sentiment.\n"
    "An aspect term is a word or phrase copied verbatim from the sentence that names something being "
    "evaluated.\n"
    "sentiment is exactly one of: positive, negative, neutral, conflict "
    "(conflict = both positive and negative).\n"
    "If the sentence contains no aspect term, return an empty array.\n"
    "Return ONLY a JSON array and nothing else. "
    'Each element must be {"aspect": "<term>", "sentiment": "<sentiment>"}.'
)

def build_prompt(text, examples=None):
    parts = [SYSTEM_INSTRUCTION, ""]
    if examples:
        for ex in examples:
            parts.append(f'Sentence: "{ex["text"]}"')
            parts.append("Output: " + json.dumps(
                [{"aspect": p["aspect"], "sentiment": p["sentiment"]} for p in ex["aspects"]],
                ensure_ascii=False))
            parts.append("")
    parts.append(f'Sentence: "{text}"')
    parts.append("Output:")
    return "\n".join(parts)

train = json.loads((DATA_DIR / "train.json").read_text(encoding="utf-8"))
print("train sentences:", len(train))

def to_completion(s):
    return json.dumps([{"aspect": p["aspect"], "sentiment": p["sentiment"]} for p in s["aspects"]],
                      ensure_ascii=False)

def to_conversation(s):
    return {"messages": [
        {"role": "user",      "content": [{"type": "text", "text": build_prompt(s["text"])}]},
        {"role": "assistant", "content": [{"type": "text", "text": to_completion(s)}]},
    ]}

rows = [to_conversation(s) for s in train]
print("example user turn:\n", rows[0]["messages"][0]["content"][0]["text"][:200])
print("-> assistant:", rows[0]["messages"][1]["content"][0]["text"])

train sentences: 1616
example user turn:
 You are an aspect-based sentiment analysis system for customer reviews.
Given one sentence, extract every aspect term that is explicitly mentioned and its sentiment.
An aspect term is a word or phrase
-> assistant: [{"aspect": "beef", "sentiment": "positive"}]


## 4. Load Qwen3.5-2B (4-bit QLoRA) and attach LoRA

In [ ]:
from unsloth import FastVisionModel
import torch

MAX_SEQ = 1024  # ABSA inputs are short; small value saves VRAM and speeds training

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-2B",
    load_in_4bit = True, # QLoRA (matches the Gemma run)
    use_gradient_checkpointing = "unsloth",
    max_seq_length = MAX_SEQ,
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # text-only task -> freeze the vision layers
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

==((====))==  Unsloth 2026.6.1: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


model.safetensors.index.json:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

processor_config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/15.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/876 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

## 5. Format dataset (train / tiny holdout)

Same as the Gemma run: a small 5% holdout (seed 42) to watch eval loss for overfitting.

In [ ]:
from datasets import Dataset

ds = Dataset.from_list(rows)
ds = ds.train_test_split(test_size=0.05, seed=42)
print(ds)

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 1535
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 81
    })
})


## 6. Train (3 epochs)

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),  # must use for vision models
    train_dataset = ds["train"],
    eval_dataset = ds["test"],
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,      # effective batch 8
        warmup_ratio = 0.05,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 10,
        eval_strategy = "epoch",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = str(OUT_DIR / "checkpoints"),
        report_to = "none",
        # required by UnslothVisionDataCollator:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = MAX_SEQ,
    ),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Model does not have a default image size - using 512
Unsloth: Switching to float32 training since model cannot work with float16


In [ ]:
stats = trainer.train()
print(stats)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


GPU = Tesla T4. Max memory = 15.637 GB. 4.014 GB reserved before training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,535 | Num Epochs = 3 | Total steps = 576
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 16,819,200 of 2,230,060,864 (0.75% trained)


Epoch,Training Loss,Validation Loss
1,0.382935,0.339133
2,0.298287,0.343254
3,0.264392,0.362977


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/absa/finetune/qwen3_5_2b/checkpoints/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/absa/finetune/qwen3_5_2b/checkpoints/checkpoint-576/tokenizer_config.json.


TrainOutput(global_step=576, training_loss=0.35216599847707486, metrics={'train_runtime': 2065.8737, 'train_samples_per_second': 2.229, 'train_steps_per_second': 0.279, 'total_flos': 9271794657855744.0, 'train_loss': 0.35216599847707486, 'epoch': 3.0})
train_runtime: 2065.9 s (34.43 min)
peak reserved memory: 4.014 GB (25.7 % of max)


## 7. Quick sanity check

One example through the chat-template inference path, just to confirm the model
produces a JSON array before running the full test set.

In [ ]:
FastVisionModel.for_inference(model)

def generate_one(text, max_new_tokens=128):
    messages = [{"role": "user", "content": [{"type": "text", "text": build_prompt(text)}]}]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(text=input_text, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

s = train[0]["text"]
print("INPUT :", s)
print("OUTPUT:", generate_one(s))

INPUT : When you want a piece of beef, head on over.
OUTPUT: [{"aspect": "beef", "sentiment": "positive"}]


## 8. Full evaluation on the test set -> predictions JSONL

Runs every `test.json` sentence through the fine-tuned model and writes one record per sentence to the JSONL file, in the same schema as the Gemma fine-tuned predictions.

In [ ]:
import json, re, time

PRED_PATH = PRED_DIR / "qwen3_5_2b_finetuned__finetuned.jsonl"

test = json.loads((DATA_DIR / "test.json").read_text(encoding="utf-8"))
print("test sentences:", len(test))

def gold_of(item):
    return [{"aspect": p["aspect"], "sentiment": p["sentiment"]} for p in item.get("aspects", [])]

def parse_aspects(raw):
    if not raw:
        return None
    s = raw.strip()
    s = re.sub(r"^```(?:json)?", "", s).strip()
    s = re.sub(r"```$", "", s).strip()
    i, j = s.find("["), s.rfind("]")
    if i == -1 or j == -1 or j < i:
        return None
    try:
        data = json.loads(s[i:j+1])
    except Exception:
        return None
    if not isinstance(data, list):
        return None
    out = []
    for d in data:
        if isinstance(d, dict) and "aspect" in d and "sentiment" in d:
            out.append({"aspect": str(d["aspect"]), "sentiment": str(d["sentiment"])})
    return out

done = set()
if PRED_PATH.exists():
    for line in PRED_PATH.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            try: done.add(json.loads(line)["id"])
            except Exception: pass
print("already done:", len(done))

n_ok = 0
with PRED_PATH.open("a", encoding="utf-8") as f:
    for idx, item in enumerate(test):
        _id = item.get("id", idx)
        if _id in done:
            continue
        t0 = time.perf_counter()
        raw = generate_one(item["text"])
        dt = time.perf_counter() - t0

        pred = parse_aspects(raw)
        ok = pred is not None
        mem_mb = round(torch.cuda.max_memory_reserved() / 1024 / 1024, 1)
        record = {
            "id": _id,
            "text": item["text"],
            "gold": gold_of(item),
            "pred": pred if ok else [],
            "ok": ok,
            "time": dt,
            "mem_mb": mem_mb,
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()
        n_ok += int(ok)
        if (idx + 1) % 25 == 0:
            print(f"{idx+1}/{len(test)}  compliance so far ~ {n_ok}/{idx+1}")

print("done. wrote predictions to", PRED_PATH)

test sentences: 405
already done: 0
25/405  compliance so far ~ 25/25
50/405  compliance so far ~ 50/50
75/405  compliance so far ~ 75/75
100/405  compliance so far ~ 100/100
125/405  compliance so far ~ 125/125
150/405  compliance so far ~ 150/150
175/405  compliance so far ~ 175/175
200/405  compliance so far ~ 200/200
225/405  compliance so far ~ 225/225
250/405  compliance so far ~ 250/250
275/405  compliance so far ~ 275/275
300/405  compliance so far ~ 300/300
325/405  compliance so far ~ 325/325
350/405  compliance so far ~ 350/350
375/405  compliance so far ~ 375/375
400/405  compliance so far ~ 400/400
done. wrote predictions to /content/drive/MyDrive/absa/predictions/qwen3_5_2b_finetuned__finetuned.jsonl


In [ ]:
lines = [json.loads(l) for l in PRED_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
n = len(lines)
ok = sum(1 for r in lines if r["ok"])
avg_t = sum(r["time"] for r in lines) / max(n, 1)
print(f"records: {n}")
print(f"parse compliance: {ok}/{n} = {ok/max(n,1):.1%}")
print(f"avg time/example (transformers+NF4, NOT comparable to Ollama): {avg_t:.2f} s")
print("sample:", lines[0])

records: 405
parse compliance: 405/405 = 100.0%
avg time/example (transformers+NF4, NOT comparable to Ollama): 4.35 s
sample: {'id': '1386', 'text': "The food's dazzling flavors overwhelm the palate, truly embracing the beauty of authentic Thai cuisine.", 'gold': [{'aspect': 'food', 'sentiment': 'positive'}, {'aspect': 'Thai cuisine', 'sentiment': 'positive'}, {'aspect': 'flavors', 'sentiment': 'positive'}], 'pred': [{'aspect': "food's flavors", 'sentiment': 'positive'}, {'aspect': 'cuisine', 'sentiment': 'positive'}], 'ok': True, 'time': 4.71673218900014, 'mem_mb': 3828.0}
